# STOP-Collapse in World Model Navigation

**Finding:** When using DreamerV3/R2-Dreamer for Habitat ObjectNav with STOP as a terminal action, the agent collapses to always calling STOP on step 1, regardless of reward shaping, entropy bonuses, or gradient fixes.

This notebook documents the progressive debugging of this failure mode and explains why DreamerNav-style STOP-as-no-op is necessary.

| Run | Configuration | Result |
|-----|--------------|--------|
| 1 | Default (`act_entropy=3e-4`) | Collapse by step ~2k |
| 2 | Higher entropy (`act_entropy=3e-2`) | Collapse by step ~5k |
| 3 | + Stop penalty (`-1.0`) | Collapse by step ~3k |
| 4 | + No-STOP prefill | Collapse by step ~5k |
| 5 | + Imagination weight fix | Collapse by step ~5k |

In [ ]:
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 4)})

LOG_DIR = Path("../../../output/r2dreamer-habitat-baseline/stop-collapse-logs")

In [ ]:
def parse_log(path):
    """Extract episode lengths and loss values from training stdout."""
    episodes = []  # (step, ep_num, reward, steps)
    losses = []    # (step, total, dyn, rew, policy)

    ep_pat = re.compile(
        r"\[step\s+(\d+)\] episode (\d+): "
        r"reward=([\-\d.]+) success=([\d.]+) "
        r"spl=([\d.]+) steps=(\d+)"
    )
    loss_pat = re.compile(
        r"\[step\s+(\d+)/\d+\] "
        r"total=([\-\d.]+) dyn=([\-\d.]+) "
        r"rew=([\-\d.]+) policy=([\-\d.]+) fps=(\d+)"
    )

    text = Path(path).read_text()
    for m in ep_pat.finditer(text):
        episodes.append({
            "step": int(m.group(1)),
            "episode": int(m.group(2)),
            "reward": float(m.group(3)),
            "success": float(m.group(4)),
            "spl": float(m.group(5)),
            "ep_steps": int(m.group(6)),
        })
    for m in loss_pat.finditer(text):
        losses.append({
            "step": int(m.group(1)),
            "total": float(m.group(2)),
            "dyn": float(m.group(3)),
            "rew": float(m.group(4)),
            "policy": float(m.group(5)),
            "fps": int(m.group(6)),
        })
    return episodes, losses

In [ ]:
runs = {
    "Run 1: Default (ent=3e-4)": "run1_no_penalty.log",
    "Run 2: Higher entropy (3e-2)": "run2_entropy_3e2.log",
    "Run 3: + Stop penalty (-1.0)": "run3_stop_penalty.log",
    "Run 4: + No-STOP prefill": "run4_no_stop_prefill.log",
    "Run 5: + Weight fix": "run5_weight_fix.log",
}

data = {}
for name, fname in runs.items():
    path = LOG_DIR / fname
    if path.exists():
        eps, losses = parse_log(path)
        data[name] = {"episodes": eps, "losses": losses}
        print(f"{name}: {len(eps)} episodes, {len(losses)} loss logs")
    else:
        print(f"{name}: NOT FOUND")

## Episode Length Over Training

The key diagnostic: mean episode length should be >>1 for a healthy agent. Collapse to STOP shows as episode length → 1.

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(4 * len(data), 4), sharey=True)
if len(data) == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, data.items()):
    eps = d["episodes"]
    if not eps:
        continue
    steps = [e["step"] for e in eps]
    lengths = [e["ep_steps"] for e in eps]

    ax.scatter(steps, lengths, s=1, alpha=0.3)
    # Rolling mean over 50 episodes
    if len(lengths) >= 50:
        import pandas as pd
        rolling = pd.Series(lengths).rolling(50).mean().values
        ax.plot(steps, rolling, color="red", linewidth=2, label="rolling 50")
    ax.set_xlabel("Step")
    ax.set_title(name, fontsize=9)
    ax.set_ylim(0, 50)
    ax.axhline(y=1, color="black", linestyle="--", alpha=0.5, label="collapse")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Episode Length")
fig.suptitle("STOP-Collapse: Episode Length → 1 Across All Configurations",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Episode Length Distribution

Histogram showing how many episodes have length 1 (STOP on first step).

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(3.5 * len(data), 3.5), sharey=True)
if len(data) == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, data.items()):
    eps = d["episodes"]
    if not eps:
        continue
    lengths = [e["ep_steps"] for e in eps]
    pct_1 = sum(1 for l in lengths if l == 1) / len(lengths) * 100

    ax.hist(lengths, bins=range(1, 52), edgecolor="black", linewidth=0.3)
    ax.set_xlabel("Episode Length")
    ax.set_title(f"{name}\n({pct_1:.0f}% are length 1)", fontsize=9)
    ax.set_xlim(0, 50)
    ax.grid(True, alpha=0.3, axis="y")

axes[0].set_ylabel("Count")
fig.suptitle("Episode Length Distributions", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Policy Loss Across Runs

The policy loss shows how much the actor is being updated. Near zero = no learning signal.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for name, d in data.items():
    losses = d["losses"]
    if not losses:
        continue
    steps = [l["step"] for l in losses]
    policy = [l["policy"] for l in losses]
    ax.plot(steps, policy, linewidth=1.5, label=name)

ax.axhline(y=0, color="black", linestyle="-", alpha=0.3)
ax.set_xlabel("Step")
ax.set_ylabel("Policy Loss")
ax.set_title("Policy Loss: Higher Entropy Produces Stronger Signal, But Still Insufficient")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Root Cause Analysis

### Why does the agent always STOP?

Three interacting mechanisms create a self-reinforcing collapse:

1. **Random STOP probability**: With 4 actions and uniform initialization, STOP has 25% probability. Expected episode length = 4 steps. The replay buffer fills with very short episodes.

2. **Dead gradient from continuation weighting**: In DreamerV3's imagination, the policy gradient is weighted by `cumprod(cont * disc)`. When STOP → terminal (`cont ≈ 0`), the weight goes to zero, eliminating the gradient signal. The actor receives *no learning signal* for STOP actions. (Run 5 attempted to fix this by shifting weights, but the pessimistic critic trap remained.)

3. **Pessimistic critic trap**: The critic trains on experience dominated by 1-step STOP episodes. It learns that all states have value ≈ reward_of_STOP. In imagination, non-STOP actions bootstrap from this pessimistic critic, producing returns worse than STOP. The actor rationally prefers STOP under these value estimates.

### Attempted fixes and why they failed

| Fix | Mechanism | Why It Failed |
|-----|-----------|---------------|
| Higher entropy (3e-2) | Force exploration via entropy bonus | Too weak vs. advantage signal; 25% STOP probability still fills buffer with short episodes |
| Stop penalty (-1.0) | Negative reward for unsuccessful STOP | Dead gradient: continuation weight zeros out penalty signal in imagination |
| No-STOP prefill | Seed buffer with multi-step episodes | Good data diluted by STOP episodes once training starts |
| Weight fix (step 0 = 1.0) | Ensure gradient flows for first imagined step | Fixes dead gradient but pessimistic critic still favors STOP |

### Solution: DreamerNav-style STOP

**Make STOP a no-op** (no movement, no episode termination). Episodes end only via:
- Reaching the goal radius (0.1m)
- Exceeding max steps (500)

This eliminates the collapse because:
- STOP no longer triggers `done=True`, so `cont ≈ 1` and gradients flow normally
- STOP gives 0 geodesic delta reward (no movement = no progress), making it naturally suboptimal vs. MOVE_FORWARD
- The critic sees full 500-step episodes, learning meaningful value estimates

## Throughput Summary

Despite the collapse, all runs achieved stable throughput on H100.

In [ ]:
for name, d in data.items():
    losses = d["losses"]
    if not losses:
        continue
    fps_vals = [l["fps"] for l in losses]
    eps = d["episodes"]
    lengths = [e["ep_steps"] for e in eps]
    pct_1 = sum(1 for l in lengths if l == 1) / len(lengths) * 100 if lengths else 0
    print(f"{name}:")
    print(f"  FPS: {np.mean(fps_vals[-5:]):.0f} (steady state)")
    print(f"  Episodes: {len(eps)}, {pct_1:.0f}% length-1")
    print(f"  Mean ep length: {np.mean(lengths):.1f}")
    print()